Install the following python packages using your package manager of choice (pip in this case)

In [ ]:
!pip install numpy
!pip install scipy
!pip install pandas
!pip install f90nml
!pip install numba

Import the `m3dis` python package amongst other common packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from m3dis.m3dis import m3dis
plt.style.use('m3dis/Bergemann2020.mplstyle')
#plt.style.use('m3dis/Bergemann2023.mplstyle') # uncomment for serif axis labels
%config InlineBackend.figure_format='svg' # turns plots into vector graphics inside this notebook

Execute Multi by calling the Dispatch executable and supplying the default demo input parameter file `demo.nml`.
This will run a 1D solar Marcs model using a 6-Level toy Barium atom.

In [ ]:
! ./dispatch.x demo.nml

The default output directory is `./data/`. A new subfolder is created according to the name of your input parameter file. It can be opened and read as follows:

In [ ]:
run = m3dis.read('data/demo/')

Now we have created an object with the name of `run`. This object contains all of the data and meta-data of our demo run. An array accessible via `run.line` contains multiple objects of the `trans_class`. This class makes plotting of spectral lines very easy. Each run stores both LTE and NLTE spectra by default. Let's make a quick comparison:

In [ ]:
imu = None # None gives flux, use integers to iterate through angles
norm = True # set to False for unnormalised line profile
kr = 0 # line index (note that python starts counting from 0)

line = run.line[kr]
line.plot(imu=imu, norm=norm, LTE=False, linestyle='-', label='non-LTE')
line.plot(imu=imu, norm=norm, LTE=True, linestyle='--', label='LTE')
plt.legend()

In [ ]:
dlam = 0.8 # zooms into the line profile to ±dlam angstroms.
norm = False
kr = 0

line = run.line[kr]

 # iterate over all angles
for imu, mu in enumerate(run.mus):
  label = r'$\mu=${:.2f}'.format(mu)
  line.plot(imu=imu, norm=norm, dlam=dlam, label=label)

plt.ylabel('Intensity [erg/s/cm$^2$/sr/Hz]')
plt.legend()

In [ ]:
dlam = 0.5
imu = 0
kr = 0

line = run.line[kr]

 # convolve with doppler profiles
for vdop in [1.0, 3.0, 5.0]:
  label = 'v={:.1f} km/s'.format(vdop)
  line.plot(imu=imu, dlam=dlam, vdop=vdop, label=label)

plt.legend()

In [ ]:
line = run.line[kr]
line.prepare()
data = {}
data['Wavelength'] = line.lam
data['Flux'] = line.flux / line.flux_cnt

# save angle resolved line profiles as ascii table
for imu, mu in enumerate(run.mus):
  label = '{:.3f}'.format(mu)
  data[label] = line.ie[:,imu] / line.cnt[:,imu]

df = pd.DataFrame(data=data)
ascii_table = df.to_string(index=False)
print(ascii_table)